# Lab 36: The Name in the Noise

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 4, *The Language Engine*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-36.ipynb)

**What you will do:** run trials in which a friend says your name versus a neutral word while you are absorbed in a noisy conversation, compare your detection rate with the roughly one-in-three rate the book reports, and test how much background noise an open speech-recognition model can tolerate before it, too, misses a target word.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

If you use headphones or a speaker to play crowd noise, keep the volume moderate — you will be
listening to it, on and off, across several trials.

## The experiment in the book

In Lab 36 you had a conversation in a noisy environment and, without warning, a third person
said your name from across the room. Even though you were not listening to them, you had a
real chance of noticing — the **own-name effect**. Trying the same thing with a neutral word
instead of your name showed the breakthrough is rarer, confirming the effect is specific to
personally relevant sounds. The book is precise about how often this actually happens: in
classic dichotic-listening studies, roughly a third of participants noticed their own name in
a channel they were told to ignore (Moray, 1959; Wood & Cowan, 1995).

## Record your results

Run several trials with a friend or small group: during a distracting conversation or noisy
recording, have someone say either your name or a neutral word from across the room, and log
whether you noticed it, using an anonymous ID for each listener if you share this further.

In [ ]:
# example data: replace with your own trial log
trials = pd.DataFrame({
    "listener": ["you", "you", "you", "friend A", "friend A", "friend A", "friend B", "friend B"],
    "word_type": ["name", "neutral", "name", "name", "neutral", "name", "name", "neutral"],
    "noticed": [True, False, True, False, False, True, True, False],
})
display(trials)

trials.groupby("word_type").noticed.mean().plot.bar(color=["firebrick", "grey"], figsize=(5, 3))
plt.ylabel("proportion noticed"); plt.xticks(rotation=0); plt.ylim(0, 1); plt.show()

## Compare

The book reports that roughly a third of listeners notice their own name in an unattended
channel. Compare your own group's detection rate for names against that figure, and check
that the neutral-word rate is lower, as the book's Step 3 predicts.

In [ ]:
name_rate = trials.loc[trials.word_type == "name", "noticed"].mean()
neutral_rate = trials.loc[trials.word_type == "neutral", "noticed"].mean()
print(f"Your name detection rate: {name_rate:.0%}")
print(f"Neutral-word detection rate: {neutral_rate:.0%}")
if 0.2 <= name_rate <= 0.5:
    print("In the neighbourhood of the roughly one-in-three rate the book reports.")
else:
    print("Some way from the roughly one-in-three figure -- a handful of trials, or a name"
          " that is unusually common or unusual, can shift this easily.")
if neutral_rate < name_rate:
    print("Neutral words broke through less often than your name, matching the book's claim"
          " that the effect is specific to personally relevant stimuli.")

## The AI side

Smart speakers use always-on "wake-word" detection — a lightweight AI equivalent of the
brain's watchlist for personally relevant sounds. Below, Whisper (Radford et al., 2023)
transcribes a real speech clip mixed with increasing amounts of background noise, and checks
whether a target word from the clip survives in the transcription at each noise level — a
simple stand-in for how noise degrades keyword detection.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q transformers datasets soundfile

import numpy as np
from datasets import load_dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en")

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
audio = ds[0]["audio"]["array"]
sr = ds[0]["audio"]["sampling_rate"]
target_word = ds[0]["text"].split()[1].strip(",.").lower()  # a real word from the clip
print(f"Target word to detect: '{target_word}'")

def transcribe_at_snr(snr_db, seed=0):
    rng = np.random.default_rng(seed)
    noise = rng.normal(0, 1, len(audio))
    scale = np.sqrt(np.mean(audio ** 2) / (np.mean(noise ** 2) * 10 ** (snr_db / 10)))
    noisy = audio + scale * noise
    inputs = processor(noisy, sampling_rate=sr, return_tensors="pt")
    ids = model.generate(inputs.input_features)
    return processor.batch_decode(ids, skip_special_tokens=True)[0].lower()

rows = []
for snr_db in [20, 10, 0, -5, -10]:
    text = transcribe_at_snr(snr_db)
    rows.append({"SNR (dB)": snr_db, "transcription": text, "target word detected": target_word in text})
detection = pd.DataFrame(rows)
display(detection)

detection.set_index("SNR (dB)")["target word detected"].astype(int).plot.bar(color="steelblue", figsize=(6, 3))
plt.ylabel("target word detected (1 = yes)"); plt.title(f"Detecting '{target_word}' as noise increases"); plt.show()

As the signal-to-noise ratio (SNR) drops, Whisper's transcription should eventually stop
containing the target word — its own version of a missed breakthrough. The parallel to the
own-name effect is loose: Whisper has no watchlist of personally significant words and no
amygdala-driven salience signal; it treats every word in its vocabulary identically unless it
is given a prompt (as in Lab 29 and Lab 31). A real wake-word system is also purpose-built and
always listening at low power, quite unlike running a general transcription model once per
clip, as this demonstration does.

## Pool the class data

Pool detection rates for name versus neutral word across the class (anonymous IDs, never
names), and see whether the pooled estimate lands closer to the roughly one-in-three figure
the book cites than any single person's small number of trials could.

In [ ]:
import io
csv_text = """id,word_type,noticed
P01,name,True
P01,neutral,False
P02,name,False
P02,neutral,False
P03,name,True
P03,neutral,False
P04,name,False
P04,neutral,False
P05,name,True
P05,neutral,True
"""  # example data: replace with your class CSV
df = pd.read_csv(io.StringIO(csv_text))
name_df = df[df.word_type == "name"]

boot = [name_df.noticed.sample(len(name_df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class name-detection rate: {name_df.noticed.mean():.0%} "
      f"(95% bootstrap interval {lo:.0%}-{hi:.0%}, n = {len(name_df)})")
df.groupby("word_type").noticed.mean().plot.bar(color=["firebrick", "grey"], figsize=(5, 3))
plt.ylabel("proportion noticed"); plt.axhline(1/3, color="grey", linestyle="--", label="~1 in 3 (book)")
plt.legend(); plt.xticks(rotation=0); plt.show()

## Questions to think about

1. The book notes that people with lower working-memory capacity are, somewhat counter-intuitively, more likely to notice their name in an unattended channel (Conway, Cowan, & Bunting, 2001), apparently because they cannot keep the attentional filter as tightly closed. Does this fit or complicate your intuition that a "better" attentional system should be one that filters out distractions most completely?
2. In the AI Lab, at what SNR did the target word disappear from the transcription? Was the drop-off sudden (present, then suddenly absent) or gradual (garbled before it disappeared entirely)? How does that compare with your own experience of noticing (or not noticing) your name -- was it ever a partial, uncertain catch rather than a clean yes or no?
3. The book distinguishes "processed to a shallow level" from "consciously attended." Whisper either outputs a word or does not; it has no equivalent of a shallow, below-conscious processing stage. What would it mean, computationally, to build a system with that distinction?
4. Advertisers exploit the own-name effect at scale by inserting personal details (your name, your location) into messages sent to millions of people. Based on what you saw about how fragile Whisper's detection was to noise, what does this suggest about how reliable large-scale automated "personalisation" pipelines might be at correctly inserting a detail relevant to a specific person?

## Challenge

Repeat the experiment with a word that is personally significant to the listener but is not
their name -- a pet's name, a hometown, a favourite team. Log whether the breakthrough rate for
this word looks closer to the name condition or the neutral-word condition.

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-36.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")